### 库

In [14]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os
from tqdm import tqdm

from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *
import lightgbm as lgb
from sklearn.model_selection import KFold
from scipy.stats import rankdata

# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
SUBMISSION_FOLDER = 'temp_output' # 提交文件的保存目录

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

best_ials_params = {
    "num_factors": 58,
    "epochs": 20,
    "confidence_scaling": "log",
    "alpha": 49.99999999999999,
    "epsilon": 5.585081217734329,
    "reg": 0.0007759360926311159
}

best_slim_params = {
    "topK": 1000,
    "l1_ratio": 0.029739176029882,
    "alpha": 0.001
}

print("项目配置加载完成.")


项目配置加载完成.


### 辅助函数

In [90]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all

def load_best_model(recommender_class, urm_train, file_name, modelfile_path):
    """
    加载一个由超参数搜索脚本保存的最佳模型。
    """
    file_path = os.path.join(modelfile_path, file_name)

    print(f"--- 正在加载预训练模型: {file_name} ---")

    # 检查模型文件是否存在
    if not os.path.exists(file_path + ".zip"):
        print(f">>> 警告: 模型文件 '{file_path}.zip' 不存在!")
        print(">>> 请确保超参数优化已完成，并且文件名正确。")
        return None

    # 1. 初始化一个“空”的模型对象
    recommender_instance = recommender_class(urm_train)

    # 2. 调用 .load_model() 方法加载数据
    recommender_instance.load_model(folder_path=modelfile_path, file_name=file_name)

    print("模型加载成功！\n")
    return recommender_instance

def safe_min_max_scale(scores):
    """
    一个健壮的 Min-Max 归一化函数。
    如果所有分数都相同，则返回一个全零数组。
    这个函数应该只处理不含 -inf 的、干净的分数数组。
    """
    # 确保输入是有限数值
    if not np.all(np.isfinite(scores)):
        # 如果包含 inf 或 nan，这是一个上游错误信号，我们返回全零
        return np.zeros_like(scores, dtype=np.float32)

    min_val, max_val = scores.min(), scores.max()
    denominator = max_val - min_val

    if denominator == 0:
        return np.zeros_like(scores, dtype=np.float32)
    else:
        return (scores - min_val) / denominator

def safe_rank_scale(scores):
    """
    一个健壮的、基于排名的归一化函数 (融合了两个版本的优点)。

    1. 检查并处理非有限值 (NaN, inf)。
    2. 将分数数组转换为 [0, 1] 区间的相对排名。
    3. 对异常值免疫，能够抵抗 iALS 的负分等极端情况。
    4. 优雅地处理各种边界情况 (空数组, 全同值数组, 单元素数组)。
    """
    # 步骤 1: 健壮性检查，防止下游崩溃
    if not isinstance(scores, np.ndarray):
        scores = np.array(scores) # 确保输入是 numpy 数组

    if len(scores) == 0:
        return scores

    if not np.all(np.isfinite(scores)):
        # 如果包含 inf 或 nan，这是一个上游错误信号，我们返回全零
        # 这可以防止 rankdata 报错或产生意外行为
        return np.zeros_like(scores, dtype=np.float32)

    # 步骤 2: 处理“无信号”的边界情况
    if scores.min() == scores.max():
         return np.zeros_like(scores, dtype=np.float32)

    # 步骤 3: 计算排名 (这是核心)
    # method='average' 表示如果有并列分数，取平均排名
    ranks = rankdata(scores, method='average')

    # 步骤 4: 使用标准的排名归一化公式
    # (Rank - 1) / (N - 1)，确保结果严格落在 [0, 1] 区间
    # 排名最低的是 0.0, 排名最高的是 1.0
    return (ranks - 1.0) / (len(scores) - 1.0)

def diagnose_candidate_set_recall(recommender_1, recommender_2, urm_validation, candidate_cutoff_per_model=100):
    """
    诊断函数，用于计算召回层的“候选集召回率”。
    它回答了这个问题：“验证集中的正确答案，有多大比例被成功地放入了候选集？”

    Args:
        recommender_1: 第一个用于生成候选集的基模型 (例如 SLIM)。
        recommender_2: 第二个用于生成候选集的基模型 (例如 IALS)。
        urm_validation (sps.csr_matrix): 用于评估的验证集 URM。
        candidate_cutoff_per_model (int): 每个基模型生成候选物品的数量。

    Returns:
        float: 候选集召回率 (Candidate Set Recall)。
    """
    print("\n--- 开始诊断候选集召回率 ---")

    total_hits = 0
    total_true_items = 0

    users_to_evaluate = np.unique(urm_validation.tocoo().row)

    for user_id in tqdm(users_to_evaluate, desc="诊断召回层"):

        # 1. 获取该用户在验证集中的真实交互物品
        true_items = urm_validation[user_id].indices
        if len(true_items) == 0:
            continue

        total_true_items += len(true_items)

        # 2. 精确地模拟您在预测时生成候选集的流程
        recs_1 = recommender_1.recommend(user_id, cutoff=candidate_cutoff_per_model)
        recs_2 = recommender_2.recommend(user_id, cutoff=candidate_cutoff_per_model)
        candidate_items = np.union1d(recs_1, recs_2)

        # 3. 计算候选集“命中”了多少个真实物品
        hits = len(set(candidate_items) & set(true_items))
        total_hits += hits

    # 4. 计算最终的“捕捞率”
    candidate_set_recall = total_hits / total_true_items if total_true_items > 0 else 0

    return candidate_set_recall

### 第一次分割

In [7]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


### 第二次分割 k-fold

In [ ]:
# 定义用于存放 K-Fold 模型的文件夹名称
FOLD_MODEL_FOLDER = "k_fold_models"
# 创建文件夹，如果已存在则不报错
os.makedirs(FOLD_MODEL_FOLDER, exist_ok=True)

print(f"所有在 K-Fold 中训练的模型将被保存在 '{FOLD_MODEL_FOLDER}/' 文件夹中。")

print("\n--- 阶段一：使用 K-Fold 生成 OOF 训练数据 (带模型保存) ---")

# 我们将在完整的 urm_train (80%数据) 上进行操作
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

meta_features = []
meta_groups = []
meta_user_ids = []
meta_labels = []
urm_train_coo = URM_train.tocoo()

for fold_num, (train_index, val_index) in enumerate(kf.split(urm_train_coo.row)):
    # 使用 f-string 格式化，方便阅读
    print(f"\n{'='*20} FOLD {fold_num + 1}/5 {'='*20}")

    urm_train_fold = sps.csr_matrix((urm_train_coo.data[train_index],
                                     (urm_train_coo.row[train_index], urm_train_coo.col[train_index])),
                                    shape=URM_train.shape)

    # --- 训练并保存基模型 ---
    print(f"--- [Fold {fold_num + 1}] 正在训练基模型 ---")

    # 加载模型
    slim_filename = f"SLIM_fold_{fold_num + 1}"
    ials_filename = f"IALS_fold_{fold_num + 1}"
    recommender_slim_fold = load_best_model(
            recommender_class=SLIMElasticNetRecommender,
            urm_train=urm_train_fold, # <--- 修改这里：传入当前折的训练数据
            file_name=slim_filename,
            modelfile_path=FOLD_MODEL_FOLDER
    )

    # 加载模型 IALS
    recommender_ials_fold = load_best_model(
            recommender_class=IALSRecommender,
            urm_train=urm_train_fold, # <--- 修改这里：传入当前折的训练数据
            file_name=ials_filename,
            modelfile_path=FOLD_MODEL_FOLDER
    )

    # 3. 为当前 fold 的验证集交互生成特征
    print("正在为验证集交互生成 OOF 特征...")
    val_users = np.unique(urm_train_coo.row[val_index])
    # 上下文特征必须只基于当前折的训练数据计算，以防数据泄露
    print(f"--- [Fold {fold_num + 1}] 正在预计算上下文特征 ---")
    user_profile_sizes_fold = np.array(urm_train_fold.sum(axis=1)).ravel()
    item_popularity_fold = np.array(urm_train_fold.sum(axis=0)).ravel()

    for user_id in tqdm(val_users, desc="生成OOF特征"):

        # a) 获取该用户在当前 fold 验证集中的正样本
        user_mask_in_val = (urm_train_coo.row[val_index] == user_id)
        positive_items = urm_train_coo.col[val_index][user_mask_in_val]

        # b) 为正样本进行负采样
        num_positive = len(positive_items)
        if num_positive == 0: continue

        # ------------------- 关键升级：正确实现的困难负采样 -------------------
        # 1. 使用 _compute_item_score 获取对所有物品的原始分数
        #    这避免了 recommend 方法的复杂逻辑和潜在的ID问题
        user_scores = recommender_slim_fold._compute_item_score(np.array([user_id]))[0]

        # 2. 将用户在当前 fold 训练集中已见的物品的分数设为负无穷，以排除它们
        user_seen_in_train_fold = urm_train_fold[user_id].indices
        user_scores[user_seen_in_train_fold] = -np.inf

        # 3. 对分数进行排序，得到一个完整的物品排序列表
        #    np.argsort 从小到大排序，所以我们用 [::-1] 来反转
        ranked_item_indices = np.argsort(user_scores)[::-1]

        # 4. 从这个完整的排序列表中，安全地提取困难负样本
        #    我们排除掉真正的正样本
        hard_negatives_all = np.setdiff1d(ranked_item_indices, positive_items, assume_unique=True)

        # 5. 从排序靠前的困难负样本中进行采样
        #    我们只从前 200 个困难负样本中采样，以确保“难度”
        top_hard_negatives = hard_negatives_all[:200]

        if len(top_hard_negatives) < num_positive:
            negative_items = top_hard_negatives
        else:
            negative_items = np.random.choice(top_hard_negatives, size=num_positive, replace=False)

        if len(negative_items) == 0: continue

        items_to_score = np.concatenate([positive_items, negative_items])

        # c) 获取分数并进行特征工程
        slim_scores_full = recommender_slim_fold._compute_item_score(np.array([user_id]))[0]
        ials_scores_full = recommender_ials_fold._compute_item_score(np.array([user_id]))[0]

        slim_scores_raw = slim_scores_full[items_to_score]
        ials_scores_raw = ials_scores_full[items_to_score]

        #    a) 排名特征 (用于稳定性)
        slim_rank_norm = safe_rank_scale(slim_scores_raw)
        ials_rank_norm = safe_rank_scale(ials_scores_raw)

        #    b) Min-Max 特征 (用于保留强度信息)
        slim_minmax_norm = safe_min_max_scale(slim_scores_raw)
        ials_minmax_norm = safe_min_max_scale(ials_scores_raw)

        user_size = user_profile_sizes_fold[user_id]
        item_pops = item_popularity_fold[items_to_score]

        for i in range(len(items_to_score)):
            features = [
                slim_scores_raw[i],
                ials_scores_raw[i],
                slim_minmax_norm[i],
                ials_minmax_norm[i],
                # 交互特征
                slim_scores_raw[i] - ials_scores_raw[i],
                slim_scores_raw[i] * ials_scores_raw[i],
                slim_minmax_norm[i] - ials_minmax_norm[i],
                slim_minmax_norm[i] * ials_minmax_norm[i],
                # 上下文特征
                user_size,
                item_pops[i],
            ]
            meta_features.append(features)
            meta_labels.append(1 if i < num_positive else 0)
            meta_user_ids.append(user_id)


In [100]:
# --- 定义包含所有列名的列表 ---
all_feature_columns  = [
    'slim_raw', 'ials_raw', 'slim_minmax', 'ials_minmax',
    'raw_diff', 'raw_prod', 'minmax_diff', 'minmax_prod',
    'user_size', 'item_pop'
]
# --- 组合最终的元模型训练数据 ---
X_train_meta_full = pd.DataFrame(meta_features, columns=all_feature_columns )
X_train_meta_full['user_id'] = meta_user_ids
y_train_meta = pd.Series(meta_labels, name='label')

# 保存到 CSV 格式
X_train_meta_full.to_csv("X_train_meta_full.csv", index=False)
y_train_meta.to_frame().to_csv("y_train_meta.csv", index=False)

print(f"\nOOF 特征生成完毕！元模型训练数据维度: {X_train_meta_full.shape} and {y_train_meta.shape}")


OOF 特征生成完毕！元模型训练数据维度: (4850484, 11) and (4850484,)


### 训练元模型

In [106]:
# --- 从硬盘加载“主特征表” ---
X_train_meta_full = pd.read_csv("X_train_meta_full.csv")
y_train_meta = pd.read_csv("y_train_meta.csv")['label']

# 使用 pandas 的 groupby 和 size() 来计算每个用户组的大小
print("正在重建 group 数组...")
group_train = X_train_meta_full.groupby('user_id').size().to_numpy()

# 准备最终的、用于训练的特征矩阵
X_train_meta_trim = X_train_meta_full.drop(columns=['user_id'])
print("重建完成！")
print(f"  - 特征: {X_train_meta_trim.shape}")
print(f"  - 标签: {X_train_meta_trim.shape}")
print(f"  - 分组: {group_train.shape}, 共 {len(group_train)} 个用户组")
print(f"  - 检查: 特征总行数 ({len(X_train_meta_trim)}) == 分组大小总和 ({group_train.sum()}) -> {len(X_train_meta_trim) == group_train.sum()}")

正在重建 group 数组...
重建完成！
  - 特征: (4868892, 10)
  - 标签: (4868892, 10)
  - 分组: (27095,), 共 27095 个用户组
  - 检查: 特征总行数 (4868892) == 分组大小总和 (4868892) -> True


In [107]:
feature_columns = ['slim_minmax', 'ials_minmax', 'minmax_diff', 'minmax_prod']
X_train_meta = X_train_meta_trim[feature_columns]

print("\n--- 训练元模型 LGBMRanker ---")

X_train_fit = X_train_meta
y_train_fit = y_train_meta
group_fit = group_train
X_val = X_train_meta
y_val = y_train_meta
group_val = group_train

ranker = lgb.LGBMRanker(
    objective="lambdarank",     # <-- 使用 LambdaRank 算法
    metric="ndcg",              # <-- 监控 NDCG 指标
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=20,
    max_depth=5,
    reg_alpha=0.1,
    reg_lambda=0.1,
    min_child_samples=30,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# 使用早停法来找到最佳迭代次数
ranker.fit(
    X=X_train_fit,
    y=y_train_fit,
    group=group_fit,            # <-- 传入训练集的 group 信息
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],     # <-- 传入验证集的 group 信息
    eval_metric="ndcg",         # <-- 明确评估指标
    callbacks=[lgb.early_stopping(20, verbose=True)]
)

print("元模型训练完成！")


--- 训练元模型 LGBMRanker ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011907 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 4868892, number of used features: 4
Training until validation scores don't improve for 20 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

### 本地验证

In [109]:
# 定义模型所在的文件夹
COMBINE_MODEL_FOLDER = "temp_output"

print("--- 正在加载本地验证的模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_slim_local_full = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="model_recommender_slim_80p",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 IALSRecommender
recommender_ials_local_full = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="model_recommender_ials_80p",
    modelfile_path=COMBINE_MODEL_FOLDER
)

candidate_recall = diagnose_candidate_set_recall(
    recommender_1=recommender_slim_local_full,
    recommender_2=recommender_ials_local_full,
    urm_validation=URM_validation,
    candidate_cutoff_per_model=50
)

print(f"\n--- 诊断结果 ---")
print(f"每个模型召回 {50} 个物品时，候选集的总召回率为: {candidate_recall:.5f}")

--- 正在加载本地验证的模型... ---
--- 正在加载预训练模型: model_recommender_slim_80p ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputmodel_recommender_slim_80p'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: IALSRecommender_best_model ---
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model'


诊断召回层:   1%|          | 145/27058 [00:00<00:18, 1437.79it/s]

IALSRecommender: Loading complete
模型加载成功！


--- 开始诊断候选集召回率 ---


诊断召回层: 100%|██████████| 27058/27058 [00:32<00:00, 838.06it/s] 


--- 诊断结果 ---
每个模型召回 50 个物品时，候选集的总召回率为: 0.42755


In [110]:
print("\n--- 在本地验证集上评估 Stacking 模型 ---")
print("用于评估的基模型已准备就绪。")

# 开始评估
cumulative_recall = 0
num_eval_users = 0
users_to_evaluate = np.unique(URM_validation.tocoo().row)

# 预先计算好上下文特征
user_profile_sizes = np.array(URM_train.sum(axis=1)).ravel()
item_popularity = np.array(URM_train.sum(axis=0)).ravel()

for user_id in tqdm(users_to_evaluate, desc="正在本地评估"):

    true_items = URM_validation[user_id].indices
    if len(true_items) == 0:
        continue

    num_eval_users += 1

    # a. 生成候选物品 (不变)
    slim_recs = recommender_slim_local_full.recommend(user_id, cutoff=50)
    ials_recs = recommender_ials_local_full.recommend(user_id, cutoff=50)
    candidate_items = np.union1d(slim_recs, ials_recs)

    if len(candidate_items) == 0:
        continue

    # b. 获取原始分数并正确提取
    slim_scores_full = recommender_slim_local_full._compute_item_score(np.array([user_id]))[0]
    ials_scores_full = recommender_ials_local_full._compute_item_score(np.array([user_id]))[0]

    slim_scores_raw = slim_scores_full[candidate_items]
    ials_scores_raw = ials_scores_full[candidate_items]

    # c. 进行与训练时完全相同的特征工程
    #    1) 排名特征 (用于稳定性)
    slim_rank_norm = safe_rank_scale(slim_scores_raw)
    ials_rank_norm = safe_rank_scale(ials_scores_raw)

    #    2) Min-Max 特征 (用于保留强度信息)
    slim_minmax_norm = safe_min_max_scale(slim_scores_raw)
    ials_minmax_norm = safe_min_max_scale(ials_scores_raw)

    user_size_feature = user_profile_sizes[user_id]
    item_pop_features = item_popularity[candidate_items]

    # d. 创建与训练时完全相同的特征 DataFrame
    X_test_meta = pd.DataFrame({
        'slim_minmax': slim_minmax_norm,
        'ials_minmax': ials_minmax_norm,
        'minmax_diff': slim_minmax_norm - ials_minmax_norm, # 交互特征1
        'minmax_prod': slim_minmax_norm * ials_minmax_norm, # 交互特征2
        #'user_size': user_size_feature, # 对于该用户的所有候选物品，这个值是相同的
        #'item_pop': item_pop_features
    })

    # e. 使用元模型得到最终分数
    final_scores = ranker.predict(X_test_meta)

    # f. 排序并推荐
    top_local_indices = np.argsort(final_scores)[::-1][:EVALUATION_CUTOFF]
    recommended_items = candidate_items[top_local_indices]

    # g. 计算 Recall
    hits = len(set(recommended_items) & set(true_items))
    recall_at_20 = hits / len(true_items)
    cumulative_recall += recall_at_20

# 计算并报告最终的本地验证分数
final_avg_recall = cumulative_recall / num_eval_users if num_eval_users > 0 else 0

print("\n--- 本地评估完成 ---")
print(f"Stacking 模型在本地验证集上的 Recall@20: {final_avg_recall:.5f}")

正在本地评估:   0%|          | 26/27058 [00:00<01:44, 258.69it/s]


--- 在本地验证集上评估 Stacking 模型 ---
用于评估的基模型已准备就绪。


正在本地评估: 100%|██████████| 27058/27058 [01:48<00:00, 248.73it/s]


--- 本地评估完成 ---
Stacking 模型在本地验证集上的 Recall@20: 0.29037


### 最终预测与提交

In [112]:
print("\n--- 最终预测与提交 ---")

# 1. 在完整的 `urm_all` 数据上重新训练基模型，以获得最强的预测性能
print("在全部 `urm_all` 数据上训练最终的基模型...")

# 加载 SLIMElasticNetRecommender
recommender_slim_full = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 加载 IALSRecommender
recommender_ials_full = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="16-final_model_IALSRecommender-less-epochs",
    modelfile_path=MODEL_FOLDER
)

print("最终的基模型已准备就绪。")

# 2. 读取目标用户 (不变)
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

submission = []

# 3. 为每个目标用户生成推荐
for user_id in tqdm(target_user_ids, desc="生成最终推荐"):

    # a. 生成候选物品集 (同步：cutoff=50)
    slim_recs = recommender_slim_full.recommend(user_id, cutoff=50)
    ials_recs = recommender_ials_full.recommend(user_id, cutoff=50)
    candidate_items = np.union1d(slim_recs, ials_recs)

    if len(candidate_items) == 0:
        submission.append((user_id, ''))
        continue

    # ------------------- 关键同步区域开始 -------------------

    # b. 获取原始分数并正确提取
    slim_scores_full = recommender_slim_full._compute_item_score(np.array([user_id]))[0]
    ials_scores_full = recommender_ials_full._compute_item_score(np.array([user_id]))[0]

    slim_scores_raw = slim_scores_full[candidate_items]
    ials_scores_raw = ials_scores_full[candidate_items]

    # c. 进行与本地验证完全相同的特征工程
    #    注意：这里我们只计算了 minmax 相关的特征，因为本地验证只用了它们
    #    (尽管变量名可能叫 _norm，但它们是基于 safe_rank_scale 计算的，根据您的代码)
    #    我们在这里将使用与您本地验证代码完全一致的逻辑和变量名
    slim_minmax_norm = safe_rank_scale(slim_scores_raw) # 假设 safe_rank_scale 是您最终选择的
    ials_minmax_norm = safe_rank_scale(ials_scores_raw) # 假设 safe_rank_scale 是您最终选择的

    # d. 创建与本地验证完全相同的特征 DataFrame (同步：只有4个特征)
    X_test_meta = pd.DataFrame({
        'slim_minmax': slim_minmax_norm,
        'ials_minmax': ials_minmax_norm,
        'minmax_diff': slim_minmax_norm - ials_minmax_norm,
        'minmax_prod': slim_minmax_norm * ials_minmax_norm
    })

    # ------------------- 关键同步区域结束 -------------------

    # e. 使用元模型得到最终分数 (同步：使用 .predict())
    final_scores = ranker.predict(X_test_meta) # 确保 ranker 是您训练好的 LGBMRanker

    # f. 排序并推荐
    top_local_indices = np.argsort(final_scores)[::-1][:EVALUATION_CUTOFF]
    recommended_items = candidate_items[top_local_indices]

    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 4. 保存提交文件
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv("submission_stacking_synced.csv", index=False)

print("\n同步后的提交文件 'submission_stacking_synced.csv' 已生成！")


--- 最终预测与提交 ---
在全部 `urm_all` 数据上训练最终的基模型...
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 16-final_model_IALSRecommender-less-epochs ---
IALSRecommender: Loading model from file 'model_output16-final_model_IALSRecommender-less-epochs'


生成最终推荐:   0%|          | 27/27095 [00:00<01:42, 263.37it/s]

IALSRecommender: Loading complete
模型加载成功！

最终的基模型已准备就绪。


生成最终推荐: 100%|██████████| 27095/27095 [01:25<00:00, 315.68it/s]



同步后的提交文件 'submission_stacking_synced.csv' 已生成！
